In [1]:
import pandas as pd

print("Pandas imported successfully.")

Pandas imported successfully.


In [3]:
hospital = pd.read_csv("../data/processed/hospital_overview_normalized.csv")
patient_flow = pd.read_csv("../data/processed/patient_flow_normalized.csv")
department = pd.read_csv("../data/processed/department_analytics_normalized.csv")
resource = pd.read_csv("../data/processed/resource_utilization_normalized.csv")

print("All four normalized datasets loaded successfully.")

All four normalized datasets loaded successfully.


In [4]:
print("Hospital Overview shape:", hospital.shape)
print("Patient Flow shape:", patient_flow.shape)
print("Department Analytics shape:", department.shape)
print("Resource Utilization shape:", resource.shape)

Hospital Overview shape: (500, 24)
Patient Flow shape: (500, 21)
Department Analytics shape: (500, 24)
Resource Utilization shape: (500, 24)


In [5]:
print("HOSPITAL OVERVIEW COLUMNS")
print(hospital.columns.tolist())

print("\nPATIENT FLOW COLUMNS")
print(patient_flow.columns.tolist())

print("\nDEPARTMENT ANALYTICS COLUMNS")
print(department.columns.tolist())

print("\nRESOURCE UTILIZATION COLUMNS")
print(resource.columns.tolist())

HOSPITAL OVERVIEW COLUMNS
['admission_id', 'patient_id', 'hospital_id', 'hospital_name', 'department_id', 'department_name', 'admission_date', 'discharge_date', 'admission_type', 'admission_source', 'bed_id', 'bed_type', 'patient_age', 'patient_gender', 'diagnosis', 'insurance_type', 'total_bill_amount', 'payment_status', 'discharge_status', 'patient_satisfaction_score', 'mortality_flag', 'readmission_flag', 'length_of_stay_days', 'date']

PATIENT FLOW COLUMNS
['movement_id', 'admission_id', 'patient_id', 'hospital_id', 'movement_sequence', 'movement_type', 'from_department_id', 'current_department_id', 'from_department_name', 'current_department_name', 'bed_id', 'movement_datetime', 'movement_date', 'duration_in_department_hours', 'year', 'month', 'day_of_week', 'hour_of_day', 'shift', 'is_peak_hour', 'date']

DEPARTMENT ANALYTICS COLUMNS
['date', 'hospital_id', 'hospital_name', 'department_id', 'department_name', 'department_type', 'total_beds', 'occupied_beds_count', 'bed_occupancy_

In [6]:
total_admissions = hospital["admission_id"].nunique()

print("Total Admissions:", total_admissions)

Total Admissions: 500


In [7]:
average_los = hospital["length_of_stay_days"].mean()

print("Average Length of Stay (days):", round(average_los, 2))

Average Length of Stay (days): 4.98


In [8]:
readmitted_patients = hospital["readmission_flag"].sum()

readmission_rate = (
    readmitted_patients / total_admissions
) * 100

print("Readmitted Patients:", readmitted_patients)
print("Readmission Rate (%):", round(readmission_rate, 2))

Readmitted Patients: 65
Readmission Rate (%): 13.0


In [9]:
mortality_count = hospital["mortality_flag"].sum()

mortality_rate = (
    mortality_count / total_admissions
) * 100

print("Mortality Count:", mortality_count)
print("Mortality Rate (%):", round(mortality_rate, 2))

Mortality Count: 7
Mortality Rate (%): 1.4


In [10]:
total_occupied_beds = department["occupied_beds_count"].sum()
total_available_beds = department["total_beds"].sum()

bed_occupancy_rate = (
    total_occupied_beds / total_available_beds
) * 100

print("Total Occupied Beds:", total_occupied_beds)
print("Total Available Beds:", total_available_beds)
print("Bed Occupancy Rate (%):", round(bed_occupancy_rate, 2))

Total Occupied Beds: 11946
Total Available Beds: 17150
Bed Occupancy Rate (%): 69.66


In [11]:
bed_resources = resource[
    resource["resource_type"].str.lower() == "bed"
]

total_beds_in_use = bed_resources["units_in_use"].sum()
total_beds_available = bed_resources["total_units_available"].sum()

bed_utilization_rate = (
    total_beds_in_use / total_beds_available
) * 100

print("Total Beds in Use:", total_beds_in_use)
print("Total Beds Available:", total_beds_available)
print("Bed Utilization Rate (%):", round(bed_utilization_rate, 2))

Total Beds in Use: 3077
Total Beds Available: 4913
Bed Utilization Rate (%): 62.63


In [12]:
total_bill_amount = hospital["total_bill_amount"].sum()

print("Total Bill Amount:", round(total_bill_amount, 2))

Total Bill Amount: 2270434.94


In [13]:
average_satisfaction = hospital["patient_satisfaction_score"].mean()

print(
    "Average Patient Satisfaction Score:",
    round(average_satisfaction, 2)
)

Average Patient Satisfaction Score: 4.04


In [14]:
admissions_by_type = (
    hospital["admission_type"]
    .value_counts()
    .reset_index()
)

admissions_by_type.columns = ["admission_type", "admission_count"]

print(admissions_by_type)

  admission_type  admission_count
0      Emergency              209
1       Elective              155
2         Urgent              136


In [15]:
hospital["admission_date"] = pd.to_datetime(
    hospital["admission_date"],
    errors="coerce"
)

hospital["admission_month"] = (
    hospital["admission_date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_admissions = (
    hospital.groupby("admission_month")["admission_id"]
    .nunique()
    .reset_index()
)

monthly_admissions.columns = [
    "month",
    "admission_count"
]

print(monthly_admissions)

     month  admission_count
0  2026-01               94
1  2026-02               89
2  2026-03               77
3  2026-04               80
4  2026-05               76
5  2026-06               84


In [16]:
hospital["discharge_date"] = pd.to_datetime(
    hospital["discharge_date"],
    errors="coerce"
)

hospital["discharge_month"] = (
    hospital["discharge_date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_discharges = (
    hospital.groupby("discharge_month")["admission_id"]
    .nunique()
    .reset_index()
)

monthly_discharges.columns = [
    "month",
    "discharge_count"
]

print(monthly_discharges)

     month  discharge_count
0  2026-01               80
1  2026-02               85
2  2026-03               83
3  2026-04               77
4  2026-05               82
5  2026-06               80
6  2026-07               13


In [17]:
admissions_vs_discharges = pd.merge(
    monthly_admissions,
    monthly_discharges,
    on="month",
    how="outer"
)

admissions_vs_discharges = admissions_vs_discharges.fillna(0)

print(admissions_vs_discharges)

     month  admission_count  discharge_count
0  2026-01             94.0               80
1  2026-02             89.0               85
2  2026-03             77.0               83
3  2026-04             80.0               77
4  2026-05             76.0               82
5  2026-06             84.0               80
6  2026-07              0.0               13


In [18]:
admissions_by_department = (
    hospital.groupby("department_name")["admission_id"]
    .nunique()
    .reset_index()
)

admissions_by_department.columns = [
    "department_name",
    "admission_count"
]

print(admissions_by_department)

        department_name  admission_count
0            cardiology               59
1  emergency department               84
2      general medicine               95
3       general surgery               69
4                   icu               42
5           orthopedics               80
6            pediatrics               71


In [19]:
avg_los_by_department = (
    hospital.groupby("department_name")["length_of_stay_days"]
    .mean()
    .reset_index()
)

avg_los_by_department.columns = [
    "department_name",
    "average_los_days"
]

avg_los_by_department["average_los_days"] = (
    avg_los_by_department["average_los_days"].round(2)
)

print(avg_los_by_department)

        department_name  average_los_days
0            cardiology              4.86
1  emergency department              4.69
2      general medicine              4.71
3       general surgery              4.84
4                   icu              6.19
5           orthopedics              5.05
6            pediatrics              5.11


In [20]:
kpi_summary = {
    "Total Admissions": total_admissions,
    "Average LOS (days)": round(average_los, 2),
    "Readmission Rate (%)": round(readmission_rate, 2),
    "Mortality Rate (%)": round(mortality_rate, 2),
    "Bed Occupancy Rate (%)": round(bed_occupancy_rate, 2),
    "Bed Utilization Rate (%)": round(bed_utilization_rate, 2),
    "Total Bill Amount": round(total_bill_amount, 2),
    "Average Patient Satisfaction": round(average_satisfaction, 2)
}

kpi_summary_df = pd.DataFrame(
    list(kpi_summary.items()),
    columns=["KPI", "Value"]
)

print(kpi_summary_df)

                            KPI       Value
0              Total Admissions      500.00
1            Average LOS (days)        4.98
2          Readmission Rate (%)       13.00
3            Mortality Rate (%)        1.40
4        Bed Occupancy Rate (%)       69.66
5      Bed Utilization Rate (%)       62.63
6             Total Bill Amount  2270434.94
7  Average Patient Satisfaction        4.04


In [21]:
kpi_summary_df.to_csv(
    "../data/processed/kpi_summary.csv",
    index=False
)

print("KPI summary saved successfully.")

KPI summary saved successfully.


In [22]:
# STEP 1: Validate the six main KPIs

# 1. Total Admissions
validation_total_admissions = hospital["admission_id"].nunique()

# 2. Average Length of Stay
validation_average_los = hospital["length_of_stay_days"].mean()

# 3. Readmission Rate
validation_readmission_rate = (
    hospital["readmission_flag"].sum()
    / hospital["admission_id"].nunique()
) * 100

# 4. Mortality Rate
validation_mortality_rate = (
    hospital["mortality_flag"].sum()
    / hospital["admission_id"].nunique()
) * 100

# 5. Bed Occupancy Rate
validation_occupied_beds = department["occupied_beds_count"].sum()
validation_available_beds = department["total_beds"].sum()

validation_bed_occupancy_rate = (
    validation_occupied_beds / validation_available_beds
) * 100

# 6. Bed Utilization Rate
validation_bed_resources = resource[
    resource["resource_type"].str.lower() == "bed"
]

validation_beds_in_use = (
    validation_bed_resources["units_in_use"].sum()
)

validation_beds_available = (
    validation_bed_resources["total_units_available"].sum()
)

validation_bed_utilization_rate = (
    validation_beds_in_use / validation_beds_available
) * 100


# Display validation results
print("KPI VALIDATION RESULTS")
print("-" * 40)

print("Total Admissions:",
      validation_total_admissions)

print("Average LOS:",
      round(validation_average_los, 2))

print("Readmission Rate:",
      round(validation_readmission_rate, 2), "%")

print("Mortality Rate:",
      round(validation_mortality_rate, 2), "%")

print("Bed Occupancy Rate:",
      round(validation_bed_occupancy_rate, 2), "%")

print("Bed Utilization Rate:",
      round(validation_bed_utilization_rate, 2), "%")

KPI VALIDATION RESULTS
----------------------------------------
Total Admissions: 500
Average LOS: 4.98
Readmission Rate: 13.0 %
Mortality Rate: 1.4 %
Bed Occupancy Rate: 69.66 %
Bed Utilization Rate: 62.63 %
